# Model Training

Covers four runs:
- **Baseline**: raw features, default Logistic Regression + XGBoost
- **Engineered**: log transform, freq/target encoding, interactions, age features → XGBoost
- **Tuned**: engineered features + Optuna hyperparameter tuning → XGBoost

## 1. Setup

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay

import mlflow
import mlflow.sklearn
import mlflow.xgboost

from src.data import load_config, load_raw, clean, split
from src.features import build_feature_matrix
from src.model import train_baseline, tune_hyperparameters, train_model, evaluate

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 4)

cfg = load_config("../config.yaml")

## 2. Load & Prepare Data

In [ ]:
df = clean(load_raw("../data/raw"))
train_df, test_df = split(df, test_size=cfg["data"]["test_size"], random_state=cfg["split"]["random_state"])

TARGET   = cfg["data"]["target_col"]
DROP_COLS = ["id", TARGET]

X_train = train_df.drop(columns=DROP_COLS)
y_train = train_df[TARGET]
X_test  = test_df.drop(columns=DROP_COLS)
y_test  = test_df[TARGET]

print(f"Features: {X_train.columns.tolist()}")
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

## 3. Baseline Models

In [ ]:
baselines = train_baseline(X_train, y_train)

lr_probs  = baselines["logistic_regression"].predict_proba(X_test)[:, 1]
xgb_probs = baselines["xgboost_baseline"].predict_proba(X_test)[:, 1]

model_probs = [
    ("Logistic Regression", lr_probs,  baselines["logistic_regression"]),
    ("XGBoost (baseline)",  xgb_probs, baselines["xgboost_baseline"]),
]

results = [evaluate(name, y_test, probs) for name, probs, _ in model_probs]

results_df = pd.DataFrame(results).set_index("model")
results_df

## 4. Precision-Recall & ROC Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

models = [
    ("Logistic Regression", lr_probs),
    ("XGBoost (baseline)",  xgb_probs),
]

for name, probs in models:
    PrecisionRecallDisplay.from_predictions(
        y_test, probs, name=name, ax=axes[0]
    )
    RocCurveDisplay.from_predictions(
        y_test, probs, name=name, ax=axes[1]
    )

# Baseline: random classifier PR line
axes[0].axhline(y_test.mean(), color="grey", linestyle="--", label=f"Random ({y_test.mean():.2f})")
axes[0].set_title("Precision-Recall Curve")
axes[1].set_title("ROC Curve")
axes[0].legend()
axes[1].legend()
plt.tight_layout()

## 5. XGBoost Feature Importance

In [ ]:
xgb_model   = baselines["xgboost_baseline"]
importances = pd.Series(xgb_model.feature_importances_, index=X_train.columns).sort_values()

plt.figure(figsize=(8, 5))
colors = ["#DD8452" if v == importances.max() else "#4C72B0" for v in importances.values]
plt.barh(importances.index, importances.values, color=colors)
plt.title("XGBoost Feature Importance (baseline)")
plt.xlabel("Importance score")
plt.tight_layout()

## 6. Baseline Summary

In [ ]:
print("=" * 45)
print("BASELINE RESULTS (raw features, no tuning)")
print("=" * 45)
print(results_df.to_string())

## 7. Engineered Features — XGBoost

In [ ]:
train_eng, test_eng = build_feature_matrix(
    train_df, test_df,
    target_col=cfg["data"]["target_col"],
)

TARGET    = cfg["data"]["target_col"]
DROP_COLS = ["id", TARGET]

X_train_eng = train_eng.drop(columns=DROP_COLS)
X_test_eng  = test_eng.drop(columns=DROP_COLS)

print(f"Engineered features ({X_train_eng.shape[1]}): {X_train_eng.columns.tolist()}")

In [ ]:
eng_baselines = train_baseline(X_train_eng, y_train)
xgb_eng_probs = eng_baselines["xgboost_baseline"].predict_proba(X_test_eng)[:, 1]
eng_result    = evaluate("XGBoost (engineered)", y_test, xgb_eng_probs)

eng_results_df = pd.DataFrame([eng_result]).set_index("model")
eng_results_df

## 8. Feature Importance — Engineered XGBoost

In [ ]:
xgb_eng_model = eng_baselines["xgboost_baseline"]
imp_eng = pd.Series(xgb_eng_model.feature_importances_, index=X_train_eng.columns).sort_values()

plt.figure(figsize=(9, 6))
colors = ["#DD8452" if v == imp_eng.max() else "#4C72B0" for v in imp_eng.values]
plt.barh(imp_eng.index, imp_eng.values, color=colors)
plt.title("XGBoost Feature Importance (engineered)")
plt.xlabel("Importance score")
plt.tight_layout()

## 9. Comparison — Baseline vs Engineered

In [ ]:
all_results = results + [eng_result]
comparison_df = pd.DataFrame(all_results).set_index("model")
print("=" * 55)
print("RESULTS COMPARISON")
print("=" * 55)
print(comparison_df.to_string())

# PR-AUC lift
baseline_pr  = results[1]["pr_auc"]   # XGBoost baseline
engineered_pr = eng_result["pr_auc"]
lift = (engineered_pr - baseline_pr) / baseline_pr * 100
print(f"\nPR-AUC lift (XGBoost engineered vs baseline): {lift:+.1f}%")

## 10. Optuna Hyperparameter Tuning

In [ ]:
best_params = tune_hyperparameters(
    X_train_eng, y_train,
    n_trials=cfg["tuning"]["n_trials"],
    cv_folds=cfg["tuning"]["cv_folds"],
    random_state=cfg["split"]["random_state"],
)

print("\nBest hyperparameters:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

## 11. Tuned Model — Train & Evaluate

In [ ]:
tuned_model  = train_model(X_train_eng, y_train, best_params)
tuned_probs  = tuned_model.predict_proba(X_test_eng)[:, 1]
tuned_result = evaluate("XGBoost (tuned)", y_test, tuned_probs)

pd.DataFrame([tuned_result]).set_index("model")

## 12. Full Comparison

In [ ]:
all_results = results + [eng_result, tuned_result]
final_df    = pd.DataFrame(all_results).set_index("model")

print("=" * 60)
print("FULL RESULTS COMPARISON")
print("=" * 60)
print(final_df.to_string())

baseline_pr = results[1]["pr_auc"]
print(f"\nPR-AUC lift — engineered vs baseline : {(eng_result['pr_auc']  - baseline_pr) / baseline_pr * 100:+.1f}%")
print(f"PR-AUC lift — tuned vs baseline      : {(tuned_result['pr_auc'] - baseline_pr) / baseline_pr * 100:+.1f}%")

## 13. Log All Runs to MLflow

In [ ]:
mlflow.set_tracking_uri("../mlruns")
mlflow.set_experiment("cross-sell-propensity")

all_runs = [
    # (metrics,      model,                               feature_stage,  tuned)
    (results[0],     baselines["logistic_regression"],    "raw",          False),
    (results[1],     baselines["xgboost_baseline"],       "raw",          False),
    (eng_result,     eng_baselines["xgboost_baseline"],   "engineered",   False),
    (tuned_result,   tuned_model,                         "engineered",   True),
]

for metrics, model, feature_stage, tuned in all_runs:
    name = metrics["model"]
    with mlflow.start_run(run_name=name):
        mlflow.set_tags({
            "stage":    "tuned" if tuned else ("baseline" if feature_stage == "raw" else "feature-engineering"),
            "features": feature_stage,
            "tuned":    str(tuned),
            "dataset":  "health-insurance-cross-sell",
        })

        if hasattr(model, "named_steps"):           # Logistic Regression Pipeline
            clf = model.named_steps["clf"]
            mlflow.log_params({
                "model_type":   "LogisticRegression",
                "class_weight": str(clf.class_weight),
                "max_iter":     clf.max_iter,
            })
            mlflow.sklearn.log_model(model, artifact_path="model")
        else:                                       # XGBoost
            params = {
                "model_type":       "XGBoost",
                "n_estimators":     model.n_estimators,
                "max_depth":        model.max_depth,
                "learning_rate":    model.learning_rate,
                "scale_pos_weight": round(model.scale_pos_weight, 2),
            }
            if tuned:
                params.update({k: v for k, v in best_params.items()})
            mlflow.log_params(params)
            mlflow.xgboost.log_model(model, artifact_path="model")

        mlflow.log_metrics({
            "pr_auc":    metrics["pr_auc"],
            "roc_auc":   metrics["roc_auc"],
            "precision": metrics["precision"],
            "recall":    metrics["recall"],
            "threshold": metrics["threshold"],
        })

        print(f"✓ Logged: {name:<32} | features: {feature_stage:<12} | tuned: {str(tuned):<5} | PR-AUC: {metrics['pr_auc']}")

print(f"\nRuns saved to: {mlflow.get_tracking_uri()}")